<a href="https://colab.research.google.com/github/xavierloreto/ML-Aquifer-Project/blob/main/Exercicio_Acea_Aguas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, TimeSeriesSplit, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

# 1. Ler o ficheiro com o caminho correto
df = pd.read_csv('/content/Petrignano (1).csv')

# 2. Converter a coluna Date para o formato de data real
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date') # Garante que está por ordem cronológica

# 3. Imputation (Preencher buracos/zeros)
# Substituímos zeros por NaN para o ffill poder preencher
df = df.replace(0, np.nan)
df = df.ffill()

# 4. Criar os Preditores (Lags de 1 e 2 meses)
# Definimos o alvo (target) e as outras colunas (features)
target = 'Depth_to_Groundwater_P25'
features = [col for col in df.columns if col not in ['Date', target]]

# Criamos colunas com os valores dos meses anteriores
for col in features + [target]:
    df[f'{col}_lag1'] = df[col].shift(1)
    df[f'{col}_lag2'] = df[col].shift(2)

# Removemos as linhas que ficaram vazias (as primeiras duas)
df = df.dropna()

# Definimos X (o que usamos para prever) e y (o que queremos prever)
X = df[[col for col in df.columns if 'lag' in col]]
y = df[target]

print("Sucesso! Dados carregados e organizados.")
print(f"Tamanho do conjunto de dados: {df.shape}")

Sucesso! Dados carregados e organizados.
Tamanho do conjunto de dados: (4189, 19)


In [22]:
# --- 1. O HARD SPLIT ---
# Separamos 20% dos dados mais recentes para o teste final (o "exame real")
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# --- 2. AS DUAS ESTRATÉGIAS DE VALIDAÇÃO ---
# Naive: Baralha tudo (errado para séries temporais)
cv_naive = KFold(n_splits=5, shuffle=True, random_state=42)

# Temporal: Respeita a linha do tempo (correto)
cv_temporal = TimeSeriesSplit(n_splits=5)

In [23]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', DecisionTreeRegressor(max_depth=10, random_state=42))
])

# Testar com a estratégia "batoteira" (KFold)
scores_naive = cross_val_score(pipe, X_train_full, y_train_full, cv=cv_naive, scoring='r2')

# Testar com a estratégia correta (Temporal)
scores_temporal = cross_val_score(pipe, X_train_full, y_train_full, cv=cv_temporal, scoring='r2')

print(f"Naive CV R2: {scores_naive.mean():.4f}")
print(f"Temporal CV R2: {scores_temporal.mean():.4f}")

Naive CV R2: 0.9993
Temporal CV R2: 0.6069


In [24]:
def evaluate_model_selection(X_train, y_train, X_test, y_test, cv_strategy, name):
    # STEP A: Criar a "máquina" (Pipeline)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', DecisionTreeRegressor(random_state=42))
    ])

    # STEP B: Escolher parâmetros para testar (Hyperparameters)
    param_grid = {
        'regressor__max_depth': [3, 5, 10, 20] # Testamos árvores de várias profundidades
    }

    # STEP C: Procurar a melhor combinação
    grid = GridSearchCV(pipe, param_grid, cv=cv_strategy, scoring='r2')
    grid.fit(X_train, y_train)

    # STEP D: Avaliação Final
    y_pred = grid.predict(X_test)
    test_r2 = r2_score(y_test, y_pred)

    print(f"\n===== Resultados para: {name} =====")
    print(f"Melhor profundidade: {grid.best_params_}")
    print(f"Score durante o treino (CV): {grid.best_score_:.4f}")
    print(f"Score no teste real (Independente): {test_r2:.4f}")

    return grid

# Executar a comparação final
result_naive = evaluate_model_selection(X_train_full, y_train_full, X_test, y_test, cv_naive, "Naive K-Fold")
result_temporal = evaluate_model_selection(X_train_full, y_train_full, X_test, y_test, cv_temporal, "Temporal Split")


===== Resultados para: Naive K-Fold =====
Melhor profundidade: {'regressor__max_depth': 10}
Score durante o treino (CV): 0.9993
Score no teste real (Independente): 0.9804

===== Resultados para: Temporal Split =====
Melhor profundidade: {'regressor__max_depth': 20}
Score durante o treino (CV): 0.6079
Score no teste real (Independente): 0.9801
